In [1]:
import ccxt
import pandas as pd
# 更簡潔的引入方式 - 直接從套件引入類別
from src import Create_funding_fetcher,Create_klines_fetcher,FundingFetcher,KlinesFetcher
# 使用套件定義的常數
from pipeline import DataMerge, DataTransform
from backtest import Backtester
from tqdm import tqdm

In [ ]:
FundingFetcher.gateio.get_data('SC/USDT:USDT')

## Data Cleaning

### Get All Data

In [ ]:
### gateio
# FundingFetcher.gateio.get_all_data(usdt_pairs_only=True)
# KlinesFetcher.gateio.get_all_data(usdt_pairs_only=True)
# # ### okx
FundingFetcher.okx.get_all_data(usdt_pairs_only=True)
KlinesFetcher.okx.get_all_data(usdt_pairs_only=True)

### Just get symbols exist in two exhcnages

In [ ]:
shared_symbols = list(set(FundingFetcher.gateio.USDT_SYMBOLS).intersection(set(FundingFetcher.okx.USDT_SYMBOLS)))
print(f"Shared symbols length between {len(shared_symbols)}")

### Clean Data

In [ ]:
FundingFetcher.gateio._exchange_id

In [ ]:
FundingFetcher.bybit.get_all_data(usdt_pairs_only=True)

In [ ]:
KlinesFetcher.bitget.get_all_data(usdt_pairs_only=True)

In [13]:
def find_symbols_missing_klines(exchange_id):
    """
    Find symbols that have funding rate data but no klines data for a specific exchange.
    
    Parameters:
    -----------
    exchange_id : str
        The exchange identifier (e.g., 'bitget', 'okx', 'gateio', 'binance', 'bybit')
    
    Returns:
    --------
    dict : A dictionary containing:
        - 'missing_klines': list of symbols with funding rates but no klines
        - 'has_both': list of symbols with both funding rates and klines
        - 'total_funding': total number of symbols with funding rate data
        - 'total_klines': total number of symbols with klines data
    """
    import os
    from pathlib import Path
    
    # Define paths
    funding_path = Path(f'data/raw/{exchange_id}/funding_rates')
    klines_path = Path(f'data/raw/{exchange_id}/klines')
    
    # Check if directories exist
    if not funding_path.exists():
        print(f"Warning: Funding rates directory not found: {funding_path}")
        return None
    
    if not klines_path.exists():
        print(f"Warning: Klines directory not found: {klines_path}")
        return None
    
    # Get all parquet files (symbols) from each directory
    funding_files = set([f.stem for f in funding_path.glob('*.parquet')])
    klines_files = set([f.stem for f in klines_path.glob('*.parquet')])
    
    # Find symbols with funding rates but no klines
    missing_klines = sorted(list(funding_files - klines_files))
    has_both = sorted(list(funding_files & klines_files))
    
    # Create result dictionary
    result = {
        'missing_klines': missing_klines,
        'has_both': has_both,
        'total_funding': len(funding_files),
        'total_klines': len(klines_files),
        'missing_count': len(missing_klines)
    }
    
    # Print summary
    print(f"\n{'='*60}")
    print(f"Exchange: {exchange_id.upper()}")
    print(f"{'='*60}")
    print(f"Total symbols with funding rates: {result['total_funding']}")
    print(f"Total symbols with klines data: {result['total_klines']}")
    print(f"Symbols with BOTH data types: {len(has_both)}")
    print(f"Symbols MISSING klines data: {result['missing_count']}")
    print(f"{'='*60}\n")
    
    if missing_klines:
        print(f"Symbols with funding rates but NO klines data ({len(missing_klines)}):")
        print("-" * 60)
        # Print in columns for better readability
        for i in range(0, len(missing_klines), 4):
            row = missing_klines[i:i+4]
            print("  ".join(f"{s:<15}" for s in row))
        print()
    
    return result

In [17]:
# Example usage: Find symbols missing klines data for Bitget
bitget_result = find_symbols_missing_klines('bitget')


Exchange: BITGET
Total symbols with funding rates: 584
Total symbols with klines data: 577
Symbols with BOTH data types: 577
Symbols MISSING klines data: 7

Symbols with funding rates but NO klines data (7):
------------------------------------------------------------
ALPHAUSDT        BADGERUSDT       DegenRebornUSDT  LAUNCHCOINUSDT 
LOOKSUSDT        MKRUSDT          NEIROETHUSDT   



In [18]:
import time

for symbol in tqdm(bitget_result['missing_klines'], desc="Fetching missing klines for Bitget"):
    try:
        begin = time.time()
        KlinesFetcher.bitget.get_data(symbol)
        end = time.time()
        print(f"Fetched klines for {symbol} in {end - begin:.2f} seconds")
    except Exception as e:
        print(f"Error fetching klines for {symbol}: {e}")

Fetching missing klines for Bitget: 100%|██████████| 7/7 [00:00<00:00, 63.35it/s]

Error fetching klines for ALPHAUSDT: ALPHAUSDT not supported in bitget
Error fetching klines for BADGERUSDT: BADGERUSDT not supported in bitget
🚀 Fetching DegenRebornUSDT from 2023-05-31 16:00:00+00:00 to 2025-10-30 05:00:00+00:00
❌ HTTP error: 400 Client Error: Bad Request for url: https://api.bitget.com/api/v2/mix/market/history-candles?symbol=DegenRebornUSDT&productType=USDT-FUTURES&granularity=1H&startTime=1685548800000&endTime=1686265200000
✅ Finished: 0 candles in 1 API calls
Fetched klines for DegenRebornUSDT in 0.11 seconds
Error fetching klines for LAUNCHCOINUSDT: LAUNCHCOINUSDT not supported in bitget
Error fetching klines for LOOKSUSDT: LOOKSUSDT not supported in bitget
Error fetching klines for MKRUSDT: MKRUSDT not supported in bitget
Error fetching klines for NEIROETHUSDT: NEIROETHUSDT not supported in bitget


In [ ]:
# You can also check multiple exchanges
for exchange in ['bitget', 'okx', 'gateio']:
    result = find_symbols_missing_klines(exchange)
    print()

In [ ]:
# Access specific data from the result
if bitget_result:
    print("First 10 symbols missing klines data:")
    print(bitget_result['missing_klines'][:10])
    
    print("\nYou can save missing symbols to a file:")
    # import json
    # with open('missing_klines_bitget.json', 'w') as f:
    #     json.dump(bitget_result['missing_klines'], f, indent=2)

In [2]:
gateio_loader = DataTransform(FundingFetcher.gateio._exchange_id)
okx_loader = DataTransform(FundingFetcher.okx._exchange_id)
bitget_loader = DataTransform(FundingFetcher.bitget._exchange_id)

In [ ]:
gateio_loader.transform_all_symbols()
okx_loader.transform_all_symbols()
bitget_loader.transform_all_symbols()

### Merge Data

In [ ]:
merger = DataMerge('okx', 'bitget')
# merger.update_merged_data('ETHUSDT')
merged_data = merger.merge_all_symbols()

## Backtest 

### Two Exchange

In [ ]:
Backtest = Backtester(exchange1=exchange1, exchange2=exchange2)

### Individual symbol quick look

In [ ]:
Backtest.plot_symbols(symbol='SOLUSDT')

### Backtest symbol you choose

In [ ]:
portfolio_df,stat,all_symbol_df,failed_symbols = Backtest.backtest_portfolio(symbols=['SOLUSDT'],max_active_positions=1000,all_symbols=False, leverage=1, threshold=0.05,out_threshold=-0.05, value_threshold=3000000, value_out_threshold=3000000)


### Strategy Performance

In [ ]:
Backtest.plot_strategy_performance(portfolio_df)

### Backtest Symbol Details

In [ ]:
Backtest.plot_top_earning_losing(all_symbol_df, top=10)

In [ ]:
Backtest.plot_position_heatmap(portfolio_df)

In [ ]:
Backtest.portfolio_by_month(portfolio_df)

### one exchange

In [ ]:
backtest_single_exchange = Backtester(exchange1, None)

In [ ]:
backtest_single_exchange.plot_symbols(symbol='ASTER')

In [ ]:
df, symbol =  backtest_single_exchange.backtest_fundingrate('ASTER', leverage=1, threshold=0.05,out_threshold=-0.05, value_threshold=3000000, value_out_threshold=3000000)

In [ ]:
backtest_single_exchange.plot_strategy_performance(df, dpi=1000)

In [ ]:
profolio_df,stat,all_symbol_df,failed_symbols = backtest_single_exchange.backtest_portfolio(symbols=[],all_symbols=True, leverage=1, threshold=0,out_threshold=-0.100, value_threshold=30000000, value_out_threshold=30000000)

In [ ]:
backtest_single_exchange.plot_strategy_performance(profolio_df,dpi=1000)